In [1]:
import os
import re
import json
from io import BytesIO
from collections import Counter
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.storage.blob import BlobServiceClient
import networkx as nx
import plotly.graph_objects as go
import streamlit as st

# -------------------------------
# STEP 1. 환경 설정
# -------------------------------
load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    azure_endpoint=os.getenv("OPENAI_AZURE_ENDPOINT"),
    api_version=os.getenv("OPENAI_API_VERSION")
)
CHAT_DEPLOYMENT_NAME = os.getenv("CHAT_DEPLOYMENT_NAME")
STORAGE_CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
CONTAINER_NAME = "mvp-voc-data"
FILE_NAME = "서비스문의 목록.xls"

In [2]:
# -------------------------------
# STEP 2. 데이터 불러오기
# -------------------------------
@st.cache_data(ttl=600)
def load_excel():
    try:
        blob_service = BlobServiceClient.from_connection_string(STORAGE_CONNECTION_STRING)
        container_client = blob_service.get_container_client(CONTAINER_NAME)
        blob_client = container_client.get_blob_client(FILE_NAME)

        blob_data = blob_client.download_blob().readall()
        return pd.read_excel(BytesIO(blob_data), engine="xlrd")

    except Exception as e:
        st.error(f"💥 엑셀(.xls) 파일 불러오기 실패:\n{e}")
        return pd.DataFrame()


df = load_excel()

st.title("📊 VOC Insight 네트워크 분석")
st.markdown("📈 키워드 군집별로 VOC 분석이 가능합니다. ")

2025-07-23 09:15:27.061 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2025-07-23 09:15:27.062 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2025-07-23 09:15:27.062 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:15:27.535 
  command:

    streamlit run C:\Users\ktds\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-07-23 09:15:27.535 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:15:27.536 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:15:27.536 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:15:28.052 Thread 'T

DeltaGenerator()

In [3]:
# -------------------------------
# STEP 3. GPT 프롬프트 샘플
# -------------------------------
advanced_sample = """
문의 제목: "신규가입시 선호번호 조회 안되어 문의"
-> 주요 키워드: 선호번호
-> 세부 이슈: 신규가입 시 선호번호 조회 불가
-> 의미적 군집: 선호번호 조회 실패

문의 제목: "번호변경시 선호번호 조회되지 않아 확인요청 문의"
-> 주요 키워드: 선호번호
-> 세부 이슈: 번호변경 중 선호번호 조회 안 됨
-> 의미적 군집: 선호번호 조회 실패
"""

In [4]:
# -------------------------------
# STEP 4. GPT 분석 함수
# -------------------------------
def parse_gpt_response(content: str) -> dict:
    match = re.search(r"\{.*\}", content, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except:
            pass
    return {"keyword": [], "issue": [], "cluster": "기타"}

@st.cache_data(show_spinner=False)
def extract_features_gpt(title: str) -> dict:
    prompt = f"""
아래는 예시입니다:
{advanced_sample}

문의 제목: "{title}"

아래와 같은 JSON 형식으로 반환:
{{
"keyword": [...],
"issue": [...],
"cluster": "..."
}}
"""
    try:
        response = client.chat.completions.create(
            model=CHAT_DEPLOYMENT_NAME,
            messages=[{"role": "user", "content": prompt}]
        )
        return parse_gpt_response(response.choices[0].message.content.strip())
    except Exception as e:
        st.warning(f"[GPT 오류] {title}: {e}")
        return {"keyword": [], "issue": [], "cluster": "기타"}


2025-07-23 09:17:20.125 No runtime found, using MemoryCacheStorageManager


In [5]:
# -------------------------------
# STEP 5. GPT 기반 메타 카테고리 생성
# -------------------------------
def cluster_to_meta_cluster(clusters):
    prompt = f"""
다음은 VOC 군집명 목록입니다. 의미상 유사한 군집을 최대 5~10개의 메타 카테고리로 그룹화하여 아래와 같이 JSON으로 반환해주세요:

군집명 목록:
{json.dumps(list(clusters), ensure_ascii=False)}

형식 예시:
{{
  "가입/해지 이슈": ["신규가입 오류", "해지 실패"],
  "단말/유심 문제": ["유심 미인식", "단말 불량"]
}}
"""
    try:
        response = client.chat.completions.create(
            model=CHAT_DEPLOYMENT_NAME,
            messages=[{"role": "user", "content": prompt}]
        )
        content = response.choices[0].message.content.strip()
        match = re.search(r"\{.*\}", content, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        else:
            return {}
    except Exception as e:
        st.warning(f"[GPT 오류: 메타 카테고리 분류 실패] {e}")
        return {}


In [6]:
# -------------------------------
# STEP 6. 검색어 & 분석 대상 설정
# -------------------------------
search = st.sidebar.text_input("🔍 키워드 검색:", "")
sample_size = st.sidebar.slider("분석할 문의 수", 10, 500, 100)

def clean_str(s):  # 전처리
    return re.sub(r"\s+", "", str(s)).lower()

if search:
    s_clean = clean_str(search)
    df["_cleaned"] = df["문의제목"].apply(clean_str)
    filtered_df = df[df["_cleaned"].str.contains(s_clean)]
else:
    filtered_df = df.head(sample_size)

@st.cache_data(ttl=900)
def analyze_all(df_filtered):
    return [(title, extract_features_gpt(title)) for title in df_filtered["문의제목"]]

analysis_results = analyze_all(filtered_df)


2025-07-23 09:17:28.779 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:17:28.780 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:17:28.780 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:17:28.781 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:17:28.781 Session state does not function when running a script without `streamlit run`
2025-07-23 09:17:28.782 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:17:28.783 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:17:28.783 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:17

In [7]:
# -------------------------------
# STEP 7. 시각화 범위
# -------------------------------
st.sidebar.header("📐 시각화 범위 설정")
meta_limit = st.sidebar.slider("메타 카테고리 수", 1, 20, 5)
cluster_limit = st.sidebar.slider("군집 수 (메타당)", 1, 10, 4)
title_limit = st.sidebar.slider("문의 제목 수 (군집당)", 1, 20, 5)


2025-07-23 09:19:05.156 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:19:05.157 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:19:05.158 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:19:05.158 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:19:05.159 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:19:05.159 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:19:05.160 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:19:05.161 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [8]:
# -------------------------------
# STEP 8. 메타 카테고리 필터링
# -------------------------------
all_clusters = set(r["cluster"] for _, r in analysis_results)
meta_mapping = cluster_to_meta_cluster(all_clusters)

def filter_meta_by_any(search_key, meta_map, results):
    matches = set()
    for meta, clusters in meta_map.items():
        for cluster in clusters:
            for title, r in results:
                if r["cluster"] == cluster:
                    if (
                        search_key.lower() in meta.lower()
                        or search_key.lower() in cluster.lower()
                        or search_key.lower() in title.lower()
                        or any(search_key.lower() in i.lower() for i in r["issue"])
                    ):
                        matches.add(meta)
    return {m: meta_map[m] for m in matches}

meta_filtered = (
    filter_meta_by_any(search.strip(), meta_mapping, analysis_results)
    if search.strip() else meta_mapping
)

# 상위 메타 추출
cluster_title_count = Counter(r["cluster"] for _, r in analysis_results)
meta_score = {
    meta: sum(cluster_title_count.get(c, 0) for c in clusters)
    for meta, clusters in meta_filtered.items()
}
top_metas = [m for m, _ in Counter(meta_score).most_common(meta_limit)]


In [9]:
# -------------------------------
# STEP 9. 네트워크 그래프 생성 (메타 강조)
# -------------------------------
def create_network_graph(results, meta_map):
    G = nx.Graph()
    for meta in top_metas:
        G.add_node(meta, node_type="meta", frequency=50)
        clusters = meta_map[meta][:cluster_limit]

        for cluster in clusters:
            G.add_node(cluster, node_type="cluster", frequency=30)
            G.add_edge(meta, cluster)

            titles = [
                (t, r) for t, r in results if r["cluster"] == cluster
            ][:title_limit]
            for title, r in titles:
                G.add_node(title, node_type="title", frequency=10)
                G.add_edge(cluster, title)

                for issue in r["issue"]:
                    G.add_node(issue, node_type="issue", frequency=5)
                    G.add_edge(title, issue)

    # Layout 계산
    pos = nx.spring_layout(G, dim=3, seed=42, k=1.0)

    x_nodes = [pos[n][0] for n in G.nodes()]
    y_nodes = [pos[n][1] for n in G.nodes()]
    z_nodes = [pos[n][2] for n in G.nodes()]

    x_edges, y_edges, z_edges = [], [], []
    for edge in G.edges():
        x_edges += [pos[edge[0]][0], pos[edge[1]][0], None]
        y_edges += [pos[edge[0]][1], pos[edge[1]][1], None]
        z_edges += [pos[edge[0]][2], pos[edge[1]][2], None]

    # 스타일 지정
    node_color = []
    node_size = []
    node_labels = []
    node_hovertexts = []

    for n in G.nodes():
        t = G.nodes[n].get("node_type", "")
        f = G.nodes[n].get("frequency", 1)
        if t == "meta":
            node_color.append("crimson")
            node_size.append(28)
            node_labels.append(f"★ {n}")
        elif t == "cluster":
            node_color.append("orange")
            node_size.append(20)
            node_labels.append(n)
        elif t == "title":
            node_color.append("gray")
            node_size.append(12)
            node_labels.append(n)
        else:  # issue
            node_color.append("lightblue")
            node_size.append(9)
            node_labels.append(n)
        node_hovertexts.append(f"<b>{n}</b><br>유형: {t}")

    node_trace = go.Scatter3d(
        x=x_nodes, y=y_nodes, z=z_nodes,
        mode="markers+text",
        text=node_labels,
        textposition="top center",
        hovertext=node_hovertexts,
        hoverinfo="text",
        marker=dict(
            size=node_size,
            color=node_color,
            # 🚫 width는 리스트로 줄 수 없으므로 제거 (전체 굵기는 아래 한 줄로 가능)
            line=dict(
                color="black",  # 모든 노드에 테두리 색상 같게
                width=1         # 혹은 2~3 등 공통적으로 적용 가능
            )
        )
    )
    edge_trace = go.Scatter3d(
        x=x_edges, y=y_edges, z=z_edges,
        mode="lines",
        line=dict(color="lightgray", width=2),
        hoverinfo="none"
    )

    fig = go.Figure(data=[edge_trace, node_trace], layout=go.Layout(
        title="📌 GPT 기반 VOC 메타카테고리 강조 네트워크",
        width=1200, height=850,
        scene=dict(
            xaxis=dict(showbackground=False),
            yaxis=dict(showbackground=False),
            zaxis=dict(showbackground=False)
        ),
        margin=dict(l=0, r=0, b=0, t=60)
    ))
    st.plotly_chart(fig, use_container_width=True)


In [10]:
# -------------------------------
# STEP 10. 출력
# -------------------------------
if analysis_results and meta_filtered:
    st.subheader("📈 메타 카테고리 강조 네트워크")
    create_network_graph(analysis_results, meta_filtered)
else:
    st.warning("📭 유효한 결과가 없습니다 (검색 조건 확인).")


2025-07-23 09:19:18.767 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:19:18.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:19:18.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:19:18.811 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:19:18.811 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:19:18.812 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:19:18.812 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-23 09:19:18.812 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar